## 🧪 Evaluating the Supervisor Agent

In the previous notebook, we built a **multimodal Supervisor Agent** capable of:

- 📄 Understanding unstructured financial reports (via RAG)  
- 📊 Querying structured financial tables (via Genie)  
- 🧠 Orchestrating both into a single, coherent answer  

---

#### ❓ But how do we know if it actually works well?

The real challenge is **measuring its quality**:
- Is it **factually correct**?
- Does it **answer the question asked**?
- Is it **safe and reliable**?

### 📦 Setup

We install the required dependencies:

- `mlflow==3.6.0` → Enables GenAI evaluation framework  
- `databricks-openai` → Allows interaction with Databricks model endpoints  

🔁 The Python environment is restarted to ensure all packages are correctly loaded.

> 💡 This step ensures a consistent and reproducible environment for evaluation.

In [0]:
%pip install databricks-openai mlflow==3.6.0
%restart_python

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import mlflow
import mlflow.deployments
import pandas as pd
from databricks.sdk import WorkspaceClient
from mlflow.genai import evaluate
from mlflow.genai.scorers import Correctness, RelevanceToQuery, Safety

### ⚙️ Configuration

We define the key components of our evaluation setup:

- **Endpoint** → The deployed Supervisor Agent we want to evaluate  
- **Judge Model** → An LLM used to automatically score responses  
- **MLflow Experiment** → Where evaluation results will be tracked  

> Instead of manually reviewing outputs, we use an **LLM-as-a-Judge** to evaluate quality at scale.

📊 All results will be logged in **MLflow** for further analysis.

In [0]:
### ✍️ ADD YOUR SUPERVISOR ENDPOINT HERE ###
ENDPOINT_NAME = "mas-72c0c8f8-endpoint"

JUDGE_MODEL = "databricks-meta-llama-3-3-70b-instruct" # The 'AI Judge' model
EXPERIMENT_PATH = f"/Users/{dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()}/agent_evaluation"

# Initialize Workspace Client
w = WorkspaceClient()

# Set the experiment
mlflow.set_experiment(EXPERIMENT_PATH)

2026/04/09 14:41:32 INFO mlflow.tracking.fluent: Experiment with name '/Users/andrei.dmitrenko@ms.d-one.ai/agent_evaluation' does not exist. Creating a new experiment.


<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/2938543975592062', creation_time=1775745692256, experiment_id='2938543975592062', last_update_time=1775745692256, lifecycle_stage='active', name='/Users/andrei.dmitrenko@ms.d-one.ai/agent_evaluation', tags={'mlflow.experiment.sourceName': '/Users/andrei.dmitrenko@ms.d-one.ai/agent_evaluation',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'andrei.dmitrenko@ms.d-one.ai',
 'mlflow.ownerId': '145303880898045'}>

### 🎯 Evaluation Dataset

We define a **small but high-quality test set**:

- Each example includes:
  - A **user query**
  - A **reference (expected) answer**
---

### 🔄 Why this format?

In order to automate scoring MLflow expects:
- `inputs`: what we send to the model  
- `expectations`: what we compare against  

In [0]:
# Create a small test set (Query + Reference Answer)
eval_data = pd.DataFrame([
    {
        "query": "What were Avolta's reported net sales and turnover for the six months ended June 30, 2025?",
        "expected_response": "Avolta reported net sales of CHF 6,624 million and a total turnover of CHF 6,734 million for the six months ended June 30, 2025."
    },
    {
        "query": "How many treasury shares did Avolta cancel in December 2024, and what was their par value?",
        "expected_response": "In December 2024, Avolta cancelled 6,104 thousand shares, which had a par value of CHF 5.00 each."
    },
    {
        "query": "What is Avolta's medium-term leverage target?",
        "expected_response": "Avolta's medium-term leverage target is 1.5x to 2.0x net debt/CORE EBITDA, with the flexibility to go up to 2.5x."
    },
    {
        "query": "What was the CORE EBITDA for the year ended December 31, 2024?",
        "expected_response": "The CORE EBITDA for the year ended December 31, 2024, was CHF 1,267 million."
    },
    {
        "query": "How much was paid in dividends to Group shareholders during the first six months of 2025?",
        "expected_response": "During the first six months of 2025, CHF 143 million was paid in dividends to Group shareholders"
    }
])

# Format for MLflow Evaluation
eval_dataset = pd.DataFrame({
    "inputs": [{"query": q} for q in eval_data["query"]],
    "expectations": [{"expected_response": a} for a in eval_data["expected_response"]]
})

### 🔌 Prediction Function

This function is connecting the evaluation pipeline to the **live Supervisor Agent**.

---

### 🔄 What happens here?

1. Send a query to the deployed endpoint  
2. Receive a multi-step agent response  
3. Extract the **final answer text**  

In [0]:
def supervisor_predict_fn(query):
    client = mlflow.deployments.get_deploy_client("databricks")
    
    try:
        # Construct the exact payload the Agent requires
        payload = {
            "input": [{"role": "user", "content": query}]
        }
        
        # Query the endpoint directly
        response = client.predict(endpoint=ENDPOINT_NAME, inputs=payload)
        
        # Navigate the multi-turn response to find the final text
        return response["output"][-1]["content"][0]["text"]
        
    except Exception as e:
        # If an error occurs make sure the whole evaluation doesn't break
        return f"ERROR: {str(e)}"

### 📊 Running the Evaluation

We now evaluate the Supervisor Agent using **MLflow GenAI**.

---

### 🧠 The Scoring Metrics

We use three automated evaluators:

- ✅ **Correctness** → Is the answer factually accurate?  
- 🎯 **Relevance** → Does it answer the question asked?  
- 🛡️ **Safety** → Is the response appropriate and compliant?  

**Next Step:**  
Open the MLflow experiment to explore results and compare performance.

In [0]:
# Define the Scorers
correctness_metric = Correctness(model=f"endpoints:/{JUDGE_MODEL}")
relevance_metric = RelevanceToQuery(model=f"endpoints:/{JUDGE_MODEL}")
safety_metric = Safety(model=f"endpoints:/{JUDGE_MODEL}")

# Run the Evaluation
run_name=f"supervisor_eval_{pd.Timestamp.now().strftime('%H%M%S')}"
with mlflow.start_run(run_name=run_name):
    results = evaluate(
        data=eval_dataset,
        predict_fn=supervisor_predict_fn,
        scorers=[correctness_metric, relevance_metric, safety_metric],
    )

print("✅ Evaluation Complete!")

2026/04/09 14:42:06 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/5 [Elapsed: 00:00, Remaining: ?] 

/local_disk0/.ephemeral_nfs/envs/pythonEnv-1bccba20-08fd-4bea-81ae-7908ef5c088f/lib/python3.12/site-packages/mlflow/telemetry/track.py:24: FutureWarning: The legacy provider 'endpoints' is deprecated and will be removed in a futurerelease. Please update your code to use the 'databricks' provider instead.
  return func(*args, **kwargs)
/local_disk0/.ephemeral_nfs/envs/pythonEnv-1bccba20-08fd-4bea-81ae-7908ef5c088f/lib/python3.12/site-packages/mlflow/telemetry/track.py:24: FutureWarning: The legacy provider 'endpoints' is deprecated and will be removed in a futurerelease. Please update your code to use the 'databricks' provider instead.
  return func(*args, **kwargs)
/local_disk0/.ephemeral_nfs/envs/pythonEnv-1bccba20-08fd-4bea-81ae-7908ef5c088f/lib/python3.12/site-packages/mlflow/telemetry/track.py:24: FutureWarning: The legacy provider 'endpoints' is deprecated and will be removed in a futurerelease. Please update your code to use the 'databricks' provider instead.
  return func(*args,

✅ Evaluation Complete!


## 🎬 End of Part 1

In this first part of the workshop, we built a complete **Financial Intelligence Agent system** using **no-code tools**:

- 🧱 Prepared and structured data  
- 📄 Indexed unstructured documents with RAG  
- 📊 Enabled analytical querying via Genie  
- 🚦 Built a Supervisor Agent to orchestrate everything  
- 🧪 Evaluated performance using MLflow  

This is powerful and already production-ready for many use cases.

### 🤔 So why go further?

No-code tools have limitations when you need:

- **Custom routing logic** beyond simple rules  
- **Fine-grained control** over agent reasoning  
- **Custom tool integration** and chaining  
- **Advanced error handling** and **fallback strategies**  
- **Deeper observability** and **debugging**  